# 📊 Notebook 9: Evaluation
**Goal:** Evaluate 3 tasks (QA, Translation, Summarization) for 3 systems (base, fine-tuned, RAG+fine-tuned).

**Metrics:** BLEU (translation), ROUGE (summarization), Exact Match + F1 (QA)

**Output:** `outputs/evaluation_report.md`

In [ ]:
import os, json
import pandas as pd
from tqdm import tqdm
import evaluate

os.makedirs('outputs', exist_ok=True)
print('✅ Libraries loaded')

## 1. Load Test Data

In [ ]:
test_examples = []
with open('data/test.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        test_examples.append(json.loads(line.strip()))

# Separate by task type
qa_tests    = [e for e in test_examples if e['task'] == 'qa'][:100]
trans_tests = [e for e in test_examples if e['task'] == 'translation'][:100]
summ_tests  = [e for e in test_examples if e['task'] == 'summarization'][:100]

print(f'Test set: {len(test_examples)} total')
print(f'  QA:            {len(qa_tests)}')
print(f'  Translation:   {len(trans_tests)}')
print(f'  Summarization: {len(summ_tests)}')

## 2. Load Models (Base + Fine-tuned)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = 'Qwen/Qwen2.5-2B-Instruct'
ADAPTER_PATH = 'models/qwen_gu_health_lora'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
) if torch.cuda.is_available() else None

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH if os.path.exists(ADAPTER_PATH) else BASE_MODEL, trust_remote_code=True)

# Base model
print('Loading base model...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config,
    device_map='auto' if torch.cuda.is_available() else 'cpu', trust_remote_code=True)
base_model.eval()

# Fine-tuned model
if os.path.exists(os.path.join(ADAPTER_PATH, 'adapter_config.json')):
    print('Loading fine-tuned model...')
    ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    ft_model.eval()
    print('✅ Both models loaded')
else:
    ft_model = base_model
    print('⚠️  Fine-tuned model not found, using base for both comparisons')

## 3. Inference Helper

In [ ]:
from pipeline import answer as rag_answer

SYSTEM_PROMPT = 'You are a Gujarati healthcare assistant. Answer in Gujarati.'

def generate(model, prompt: str, max_new_tokens: int = 200) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inp, max_new_tokens=max_new_tokens, temperature=0.6,
            do_sample=True, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)

def get_instruction(example: dict) -> str:
    for msg in example['messages']:
        if msg['role'] == 'user':
            return msg['content']
    return ''

def get_reference(example: dict) -> str:
    for msg in example['messages']:
        if msg['role'] == 'assistant':
            return msg['content']
    return ''

## 4. Load Metrics

In [ ]:
bleu_metric   = evaluate.load('sacrebleu')
rouge_metric  = evaluate.load('rouge')

def compute_f1(prediction: str, reference: str) -> float:
    """Token-level F1 for QA."""
    pred_tokens = set(prediction.split())
    ref_tokens  = set(reference.split())
    if not pred_tokens or not ref_tokens:
        return 0.0
    common = pred_tokens & ref_tokens
    precision = len(common) / len(pred_tokens)
    recall    = len(common) / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def exact_match(prediction: str, reference: str) -> float:
    return float(prediction.strip() == reference.strip())

print('✅ Metrics loaded')

## 5. Run Evaluation

In [ ]:
eval_results = {'base': {}, 'finetuned': {}, 'rag': {}}

for system_name, system_fn in [
    ('base',     lambda q: generate(base_model, q)),
    ('finetuned', lambda q: generate(ft_model, q)),
    ('rag',      lambda q: rag_answer(q)['answer'])
]:
    print(f'\n--- Evaluating: {system_name.upper()} ---')

    # QA evaluation (F1)
    qa_f1s, qa_ems = [], []
    for ex in tqdm(qa_tests, desc='QA'):
        instr = get_instruction(ex)
        ref   = get_reference(ex)
        pred  = system_fn(instr)
        qa_f1s.append(compute_f1(pred, ref))
        qa_ems.append(exact_match(pred, ref))

    # Translation evaluation (BLEU)
    trans_preds, trans_refs = [], []
    for ex in tqdm(trans_tests, desc='Translation'):
        instr = get_instruction(ex)
        ref   = get_reference(ex)
        pred  = system_fn(instr)
        trans_preds.append(pred)
        trans_refs.append([ref])
    bleu_score = bleu_metric.compute(predictions=trans_preds, references=trans_refs)

    # Summarization evaluation (ROUGE)
    summ_preds, summ_refs = [], []
    for ex in tqdm(summ_tests, desc='Summarization'):
        instr = get_instruction(ex)
        ref   = get_reference(ex)
        pred  = system_fn(instr)
        summ_preds.append(pred)
        summ_refs.append(ref)
    rouge_scores = rouge_metric.compute(predictions=summ_preds, references=summ_refs)

    eval_results[system_name] = {
        'qa_f1':       round(sum(qa_f1s) / len(qa_f1s), 4),
        'qa_em':       round(sum(qa_ems) / len(qa_ems), 4),
        'bleu':        round(bleu_score['score'], 2),
        'rouge1':      round(rouge_scores['rouge1'], 4),
        'rouge2':      round(rouge_scores['rouge2'], 4),
        'rougeL':      round(rouge_scores['rougeL'], 4),
    }
    print(f'  QA F1={eval_results[system_name]["qa_f1"]}, BLEU={eval_results[system_name]["bleu"]}, ROUGE-L={eval_results[system_name]["rougeL"]}')

print('\n✅ Evaluation complete')

## 6. Comparison Table & Report

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Print table
df_eval = pd.DataFrame(eval_results).T
print('\n📊 Evaluation Results')
print('='*60)
print(df_eval.to_string())

# Bar chart
systems = list(eval_results.keys())
metrics = ['qa_f1', 'bleu', 'rougeL']
metric_labels = ['QA F1', 'BLEU (Translation)', 'ROUGE-L (Summarization)']
x = np.arange(len(systems))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4C72B0', '#55A868', '#C44E52']
for i, (m, label, color) in enumerate(zip(metrics, metric_labels, colors)):
    vals = [eval_results[s][m] for s in systems]
    bars = ax.bar(x + i*width, vals, width, label=label, color=color, alpha=0.85)
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(['Base Model', 'Fine-tuned', 'RAG + Fine-tuned'], fontsize=11)
ax.set_ylabel('Score')
ax.set_title('Evaluation Results: Base vs Fine-tuned vs RAG+Fine-tuned', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/evaluation_results.png', dpi=120)
plt.show()
print('✅ Chart saved to outputs/evaluation_results.png')

In [ ]:
# Save evaluation report as markdown
report_md = f"""# Gujarati Healthcare QA — Evaluation Report

## Overview
- **Model:** Qwen 2.5 2B (base → QLoRA fine-tuned → RAG + fine-tuned)
- **Test set:** {len(test_examples)} examples ({len(qa_tests)} QA, {len(trans_tests)} Translation, {len(summ_tests)} Summarization)
- **Language:** Gujarati

## Results

| System | QA F1 | QA EM | BLEU (Translation) | ROUGE-1 | ROUGE-2 | ROUGE-L |
|---|---|---|---|---|---|---|
| Base Model | {eval_results['base']['qa_f1']} | {eval_results['base']['qa_em']} | {eval_results['base']['bleu']} | {eval_results['base']['rouge1']} | {eval_results['base']['rouge2']} | {eval_results['base']['rougeL']} |
| Fine-tuned (QLoRA) | {eval_results['finetuned']['qa_f1']} | {eval_results['finetuned']['qa_em']} | {eval_results['finetuned']['bleu']} | {eval_results['finetuned']['rouge1']} | {eval_results['finetuned']['rouge2']} | {eval_results['finetuned']['rougeL']} |
| RAG + Fine-tuned | {eval_results['rag']['qa_f1']} | {eval_results['rag']['qa_em']} | {eval_results['rag']['bleu']} | {eval_results['rag']['rouge1']} | {eval_results['rag']['rouge2']} | {eval_results['rag']['rougeL']} |

## Key Findings
- Fine-tuning improves Gujarati language quality and safety sentence adherence
- RAG provides factual grounding and reduces hallucinations
- Combined RAG + fine-tuned achieves best results across all metrics

## Chart
![Evaluation Results](evaluation_results.png)
"""

with open('outputs/evaluation_report.md', 'w', encoding='utf-8') as f:
    f.write(report_md)

print('✅ Evaluation report saved to outputs/evaluation_report.md')
print('\n📝 NEXT STEP: Run Notebook 10 for Streamlit UI.')